In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("HospitalAnalytics")\
        .getOrCreate()

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
print('Spark version:', spark.version)

Spark version: 4.0.2


In [4]:
from google.colab import files
uploaded = files.upload()

Saving appointments.csv to appointments.csv
Saving patient_preferances.json to patient_preferances.json
Saving patients.csv to patients.csv


In [6]:
patients_df = spark.read.csv('patients.csv', header=True, inferSchema=True)
print('patients_df loaded')


patients_df loaded


In [7]:
appointments_df = spark.read.csv('appointments.csv', header=True, inferSchema=True)
print(' appointments_df loaded')

 appointments_df loaded


In [9]:
patients_df.printSchema()
appointments_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [10]:
print('Patients count   :', patients_df.count())
print('Appointments count:', appointments_df.count())

Patients count   : 10
Appointments count: 10


In [11]:
print('First 5 patients')
patients_df.show(5)
print('First 5 appointments')
appointments_df.show(5)

First 5 patients
+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
First 5 appointments
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+-----------

In [12]:
patients_df.select('city').distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [13]:
appointments_df.select('department').distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [14]:
patients_df.write.mode('overwrite').parquet('patients_parquet')


In [15]:
patients_parquet_df = spark.read.parquet('patients_parquet')
patients_parquet_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [16]:
print('Match:', patients_df.count() == patients_parquet_df.count())

Match: True


In [17]:
patients_df.filter(F.col('city') == 'Hyderabad').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [18]:
patients_df.filter(F.col('gender') == 'Female').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [19]:
patients_df.filter(F.col('age') > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [20]:
appointments_df.filter(F.col('status') == 'Completed').show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [21]:
appointments_df.filter(F.col('status') == 'Pending').show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [22]:
appointments_df.filter(F.col('consultation_fee') > 1500).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [23]:
patients_df.filter(F.col('insurance_status') == 'Active').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [24]:
patients_df.filter(F.col('insurance_status') == 'Inactive').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [25]:
patients_df.filter(F.col('blood_group') == 'O+').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [26]:
appointments_df.filter(F.col('department') == 'Cardiology').show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [27]:
patients_df.filter(F.col('city').isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [28]:
patients_df.filter(F.col('blood_group').isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [29]:
appointments_df.filter(F.col('consultation_fee').isNull()).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [31]:
null_counts = patients_df.select([
      F.count(F.when(F.col(c).isNull(), c)).alias(c)
          for c in patients_df.columns
          ])
null_counts.show()
null_counts_appt = appointments_df.select([
              F.count(F.when(F.col(c).isNull(), c)).alias(c)
                  for c in appointments_df.columns
                  ])
null_counts_appt.show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+

+--------------+----------+-----------+----------+----------------+----------------+------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status|
+--------------+----------+-----------+----------+----------------+----------------+------+
|             0|         0|          0|         0|               0|               1|     0|
+--------------+----------+-----------+----------+----------------+----------------+------+



In [32]:
patients_filled_city = patients_df.fillna({'city': 'Unknown'})
patients_filled_city.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [34]:
patients_filled_bg = patients_df.fillna({'blood_group': 'Not Available'})
patients_filled_bg.show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [35]:
appointments_filled_fee = appointments_df.fillna({'consultation_fee': 0})
appointments_filled_fee.show()


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [36]:
appointments_dropped = appointments_df.dropna(subset=['consultation_fee'])
print('After drop:', appointments_dropped.count())
appointments_dropped.show()

After drop: 9
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110

In [37]:
patients_dq = patients_df.withColumn(
      'data_quality_status',
          F.when(
                  F.col('city').isNull() | F.col('blood_group').isNull(),
                          'Incomplete'
                              ).otherwise('Complete')
                              )
patients_dq.show()


+----------+------------+---------+---+------+-----------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|           Complete|
|       108|  Meera Nair|    Kochi| 48|F

In [38]:
patients_dq.groupBy('data_quality_status').count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    8|
|         Incomplete|    2|
+-------------------+-----+



In [40]:
patients_df.select(F.upper('patient_name').alias('upper_name')).show()

+------------+
|  upper_name|
+------------+
|RAHUL SHARMA|
| PRIYA REDDY|
|  AMIT KUMAR|
| SNEHA PATEL|
|  FARHAN ALI|
|  NEHA SINGH|
| ARJUN VERMA|
|  MEERA NAIR|
|   KIRAN RAO|
| NISHA REDDY|
+------------+



In [41]:
patients_df.select(F.lower('patient_name').alias('lower_name')).show()

+------------+
|  lower_name|
+------------+
|rahul sharma|
| priya reddy|
|  amit kumar|
| sneha patel|
|  farhan ali|
|  neha singh|
| arjun verma|
|  meera nair|
|   kiran rao|
| nisha reddy|
+------------+



In [42]:
patients_df.select( F.length('patient_name')).show()

+--------------------+
|length(patient_name)|
+--------------------+
|                  12|
|                  11|
|                  10|
|                  11|
|                  10|
|                  10|
|                  11|
|                  10|
|                   9|
|                  11|
+--------------------+



In [44]:
patients_df.select(F.substring('patient_name', 1, 3).alias('first_3')).show()

+-------+
|first_3|
+-------+
|    Rah|
|    Pri|
|    Ami|
|    Sne|
|    Far|
|    Neh|
|    Arj|
|    Mee|
|    Kir|
|    Nis|
+-------+



In [45]:
patients_age = patients_df.withColumn(
      'age_group',
          F.when(F.col('age') < 30, 'Young')
               .when((F.col('age') >= 30) & (F.col('age') <= 50), 'Middle-aged')
                    .otherwise('Senior')
                    )
patients_age.show()


+----------+------------+---------+---+------+-----------+----------------+-----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|  age_group|
+----------+------------+---------+---+------+-----------+----------------+-----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|Middle-aged|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|      Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|Middle-aged|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|Middle-aged|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     Senior|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|Middle-aged|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|      Young|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|Middle-aged|
|       109|   Kiran Rao|Hyderab

In [46]:
patients_df.withColumn(
'insurance_flag',
 F.when(F.col('insurance_status') == 'Active', 1).otherwise(0)
  ).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|insurance_flag|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|             1|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|             1|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|             0|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|             1|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|             1|

In [47]:
patients_df.withColumn(
'senior_citizen',
  F.when(F.col('age') >= 60, 'Yes').otherwise('No')
  ).show()

+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|            No|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|            No|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|            No|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|            No|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|            No|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|            No|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|            No|

In [48]:
patients_df.select(
      F.concat_ws(' - ', F.col('patient_name'), F.col('city')).alias('name_city')
      ).show()

+--------------------+
|           name_city|
+--------------------+
|Rahul Sharma - Hy...|
|Priya Reddy - Ban...|
| Amit Kumar - Mumbai|
|Sneha Patel - Che...|
|  Farhan Ali - Delhi|
|          Neha Singh|
|  Arjun Verma - Pune|
|  Meera Nair - Kochi|
|Kiran Rao - Hyder...|
|Nisha Reddy - Ban...|
+--------------------+



In [49]:
patients_df.select('patient_name', F.trim(F.col('patient_name')).alias('trimmed_name')).show()

+------------+------------+
|patient_name|trimmed_name|
+------------+------------+
|Rahul Sharma|Rahul Sharma|
| Priya Reddy| Priya Reddy|
|  Amit Kumar|  Amit Kumar|
| Sneha Patel| Sneha Patel|
|  Farhan Ali|  Farhan Ali|
|  Neha Singh|  Neha Singh|
| Arjun Verma| Arjun Verma|
|  Meera Nair|  Meera Nair|
|   Kiran Rao|   Kiran Rao|
| Nisha Reddy| Nisha Reddy|
+------------+------------+



In [50]:
patients_df.withColumn('city', F.upper(F.col('city'))).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|
|       102| Priya Reddy|BANGALORE| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   MUMBAI| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  CHENNAI| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    DELHI| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     PUNE| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    KOCHI| 48|Female|         O-|          Active|
|       109|   Kiran Rao|HYDERABAD| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|BANGALORE| 41|Female|         A+|          Active|
+----------+

In [51]:
patients_df.groupBy('city').count().orderBy('count', ascending=False).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|Hyderabad|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
+---------+-----+



In [52]:
patients_df.groupBy('gender').count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [53]:
patients_df.groupBy('blood_group').count().show()

+-----------+-----+
|blood_group|count|
+-----------+-----+
|        AB+|    1|
|       NULL|    1|
|         O+|    2|
|         O-|    1|
|         B+|    2|
|         A+|    3|
+-----------+-----+



In [54]:
appointments_df.groupBy('department').count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [55]:
patients_df.groupBy('city').agg(F.avg('age').alias('avg_age')).show()

+---------+-------+
|     city|avg_age|
+---------+-------+
|Bangalore|   35.0|
|    Kochi|   48.0|
|  Chennai|   31.0|
|     NULL|   38.0|
|   Mumbai|   42.0|
|     Pune|   26.0|
|    Delhi|   55.0|
|Hyderabad|   34.0|
+---------+-------+



In [56]:
patients_df.groupBy('city').agg(F.max('age').alias('max_age')).show()

+---------+-------+
|     city|max_age|
+---------+-------+
|Bangalore|     41|
|    Kochi|     48|
|  Chennai|     31|
|     NULL|     38|
|   Mumbai|     42|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     35|
+---------+-------+



In [57]:
patients_df.groupBy('city').agg(F.min('age').alias('min_age')).show()

+---------+-------+
|     city|min_age|
+---------+-------+
|Bangalore|     29|
|    Kochi|     48|
|  Chennai|     31|
|     NULL|     38|
|   Mumbai|     42|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     33|
+---------+-------+



In [58]:
appointments_df.groupBy('department').agg(
      F.avg('consultation_fee').alias('avg_fee')
      ).show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [59]:
appointments_df.groupBy('department').agg(
      F.sum('consultation_fee').alias('total_fee')
      ).show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [62]:
appointments_df.groupBy('department').agg(
      F.sum('consultation_fee').alias('total_revenue')
      ).orderBy('total_revenue', ascending=False).limit(1).show()

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+



In [63]:
inner_df = patients_df.join(appointments_df, on='patient_id', how='inner')
inner_df.show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [64]:
left_df = patients_df.join(appointments_df, on='patient_id', how='left')
left_df.show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [65]:
right_df = patients_df.join(appointments_df, on='patient_id', how='right')
right_df.show()

+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|

In [66]:
full_df = patients_df.join(appointments_df, on='patient_id', how='full')
full_df.show()

+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|

In [67]:
no_appt = patients_df.join(appointments_df, on='patient_id', how='left_anti')
no_appt.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [68]:

no_patient = appointments_df.join(patients_df, on='patient_id', how='left_anti')
no_patient.show()

+----------+--------------+-----------+----------+----------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+----------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+----------+--------------+-----------+----------+----------------+----------------+---------+



In [69]:
appointments_df.groupBy('patient_id').count().alias('appointment_count').show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [70]:
fees_per_patient = appointments_df.groupBy('patient_id').agg(
      F.sum('consultation_fee').alias('total_fees')
      )
fees_per_patient.show()

+----------+----------+
|patient_id|total_fees|
+----------+----------+
|       108|      NULL|
|       101|      2500|
|       103|      1500|
|       120|      1500|
|       107|      2000|
|       102|      2000|
|       105|      1000|
|       110|      2500|
|       104|      2500|
+----------+----------+



In [72]:
fees_per_patient.join(patients_df, on='patient_id', how='inner') \
  .select('patient_id', 'patient_name', 'total_fees') \
  .orderBy('total_fees', ascending=False) \
  .limit(1).show()

+----------+------------+----------+
|patient_id|patient_name|total_fees|
+----------+------------+----------+
|       101|Rahul Sharma|      2500|
+----------+------------+----------+



In [73]:
appointments_df.groupBy('patient_id').agg(
  F.count('appointment_id').alias('appt_count')
  ).join(patients_df.select('patient_id', 'patient_name'), on='patient_id', how='left') \
  .select('patient_id', 'patient_name', 'appt_count') \
  .orderBy('appt_count', ascending=False).show()

+----------+------------+----------+
|patient_id|patient_name|appt_count|
+----------+------------+----------+
|       101|Rahul Sharma|         2|
|       108|  Meera Nair|         1|
|       103|  Amit Kumar|         1|
|       120|        NULL|         1|
|       107| Arjun Verma|         1|
|       102| Priya Reddy|         1|
|       105|  Farhan Ali|         1|
|       110| Nisha Reddy|         1|
|       104| Sneha Patel|         1|
+----------+------------+----------+



In [76]:
spending = appointments_df.groupBy('patient_id').agg(
      F.sum('consultation_fee').alias('total_fees')
      ).join(patients_df.select('patient_id', 'patient_name', 'city'), on='patient_id', how='left')
spending.show()


+----------+----------+------------+---------+
|patient_id|total_fees|patient_name|     city|
+----------+----------+------------+---------+
|       108|      NULL|  Meera Nair|    Kochi|
|       101|      2500|Rahul Sharma|Hyderabad|
|       103|      1500|  Amit Kumar|   Mumbai|
|       120|      1500|        NULL|     NULL|
|       107|      2000| Arjun Verma|     Pune|
|       102|      2000| Priya Reddy|Bangalore|
|       105|      1000|  Farhan Ali|    Delhi|
|       110|      2500| Nisha Reddy|Bangalore|
|       104|      2500| Sneha Patel|  Chennai|
+----------+----------+------------+---------+



In [77]:
w = Window.orderBy(F.desc('total_fees'))
spending.withColumn('rank', F.rank().over(w)).show()

+----------+----------+------------+---------+----+
|patient_id|total_fees|patient_name|     city|rank|
+----------+----------+------------+---------+----+
|       101|      2500|Rahul Sharma|Hyderabad|   1|
|       110|      2500| Nisha Reddy|Bangalore|   1|
|       104|      2500| Sneha Patel|  Chennai|   1|
|       107|      2000| Arjun Verma|     Pune|   4|
|       102|      2000| Priya Reddy|Bangalore|   4|
|       103|      1500|  Amit Kumar|   Mumbai|   6|
|       120|      1500|        NULL|     NULL|   6|
|       105|      1000|  Farhan Ali|    Delhi|   8|
|       108|      NULL|  Meera Nair|    Kochi|   9|
+----------+----------+------------+---------+----+



In [78]:
spending.withColumn('dense_rank', F.dense_rank().over(w)).show()

+----------+----------+------------+---------+----------+
|patient_id|total_fees|patient_name|     city|dense_rank|
+----------+----------+------------+---------+----------+
|       101|      2500|Rahul Sharma|Hyderabad|         1|
|       110|      2500| Nisha Reddy|Bangalore|         1|
|       104|      2500| Sneha Patel|  Chennai|         1|
|       107|      2000| Arjun Verma|     Pune|         2|
|       102|      2000| Priya Reddy|Bangalore|         2|
|       103|      1500|  Amit Kumar|   Mumbai|         3|
|       120|      1500|        NULL|     NULL|         3|
|       105|      1000|  Farhan Ali|    Delhi|         4|
|       108|      NULL|  Meera Nair|    Kochi|         5|
+----------+----------+------------+---------+----------+



In [79]:
spending.withColumn('row_num', F.row_number().over(w)).show()

+----------+----------+------------+---------+-------+
|patient_id|total_fees|patient_name|     city|row_num|
+----------+----------+------------+---------+-------+
|       101|      2500|Rahul Sharma|Hyderabad|      1|
|       110|      2500| Nisha Reddy|Bangalore|      2|
|       104|      2500| Sneha Patel|  Chennai|      3|
|       107|      2000| Arjun Verma|     Pune|      4|
|       102|      2000| Priya Reddy|Bangalore|      5|
|       103|      1500|  Amit Kumar|   Mumbai|      6|
|       120|      1500|        NULL|     NULL|      7|
|       105|      1000|  Farhan Ali|    Delhi|      8|
|       108|      NULL|  Meera Nair|    Kochi|      9|
+----------+----------+------------+---------+-------+



In [80]:
spending.withColumn('rank', F.rank().over(w)).filter(F.col('rank') == 1).show()

+----------+----------+------------+---------+----+
|patient_id|total_fees|patient_name|     city|rank|
+----------+----------+------------+---------+----+
|       101|      2500|Rahul Sharma|Hyderabad|   1|
|       110|      2500| Nisha Reddy|Bangalore|   1|
|       104|      2500| Sneha Patel|  Chennai|   1|
+----------+----------+------------+---------+----+



In [81]:
spending.withColumn('rank', F.rank().over(w)).filter(F.col('rank') <= 3).show()

+----------+----------+------------+---------+----+
|patient_id|total_fees|patient_name|     city|rank|
+----------+----------+------------+---------+----+
|       101|      2500|Rahul Sharma|Hyderabad|   1|
|       110|      2500| Nisha Reddy|Bangalore|   1|
|       104|      2500| Sneha Patel|  Chennai|   1|
+----------+----------+------------+---------+----+



In [83]:
w_city = Window.partitionBy('city').orderBy(F.desc('total_fees'))
spending.withColumn('city_rank', F.rank().over(w_city)) \
.filter(F.col('city_rank') == 1) \
.show()

+----------+----------+------------+---------+---------+
|patient_id|total_fees|patient_name|     city|city_rank|
+----------+----------+------------+---------+---------+
|       120|      1500|        NULL|     NULL|        1|
|       110|      2500| Nisha Reddy|Bangalore|        1|
|       104|      2500| Sneha Patel|  Chennai|        1|
|       105|      1000|  Farhan Ali|    Delhi|        1|
|       101|      2500|Rahul Sharma|Hyderabad|        1|
|       108|      NULL|  Meera Nair|    Kochi|        1|
|       103|      1500|  Amit Kumar|   Mumbai|        1|
|       107|      2000| Arjun Verma|     Pune|        1|
+----------+----------+------------+---------+---------+



In [84]:
w_city_asc = Window.partitionBy('city').orderBy(F.asc('total_fees'))
spending.withColumn('city_rank_asc', F.rank().over(w_city_asc)) \
.filter(F.col('city_rank_asc') == 1) \
.show()

+----------+----------+------------+---------+-------------+
|patient_id|total_fees|patient_name|     city|city_rank_asc|
+----------+----------+------------+---------+-------------+
|       120|      1500|        NULL|     NULL|            1|
|       102|      2000| Priya Reddy|Bangalore|            1|
|       104|      2500| Sneha Patel|  Chennai|            1|
|       105|      1000|  Farhan Ali|    Delhi|            1|
|       101|      2500|Rahul Sharma|Hyderabad|            1|
|       108|      NULL|  Meera Nair|    Kochi|            1|
|       103|      1500|  Amit Kumar|   Mumbai|            1|
|       107|      2000| Arjun Verma|     Pune|            1|
+----------+----------+------------+---------+-------------+



In [85]:
w_running = Window.orderBy('patient_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)
spending.withColumn('running_total', F.sum('total_fees').over(w_running)).show()

+----------+----------+------------+---------+-------------+
|patient_id|total_fees|patient_name|     city|running_total|
+----------+----------+------------+---------+-------------+
|       101|      2500|Rahul Sharma|Hyderabad|         2500|
|       102|      2000| Priya Reddy|Bangalore|         4500|
|       103|      1500|  Amit Kumar|   Mumbai|         6000|
|       104|      2500| Sneha Patel|  Chennai|         8500|
|       105|      1000|  Farhan Ali|    Delhi|         9500|
|       107|      2000| Arjun Verma|     Pune|        11500|
|       108|      NULL|  Meera Nair|    Kochi|        11500|
|       110|      2500| Nisha Reddy|Bangalore|        14000|
|       120|      1500|        NULL|     NULL|        15500|
+----------+----------+------------+---------+-------------+



In [86]:
w_lead = Window.orderBy('patient_id')
spending.withColumn('next_fee', F.lead('total_fees', 1).over(w_lead)).show()

+----------+----------+------------+---------+--------+
|patient_id|total_fees|patient_name|     city|next_fee|
+----------+----------+------------+---------+--------+
|       101|      2500|Rahul Sharma|Hyderabad|    2000|
|       102|      2000| Priya Reddy|Bangalore|    1500|
|       103|      1500|  Amit Kumar|   Mumbai|    2500|
|       104|      2500| Sneha Patel|  Chennai|    1000|
|       105|      1000|  Farhan Ali|    Delhi|    2000|
|       107|      2000| Arjun Verma|     Pune|    NULL|
|       108|      NULL|  Meera Nair|    Kochi|    2500|
|       110|      2500| Nisha Reddy|Bangalore|    1500|
|       120|      1500|        NULL|     NULL|    NULL|
+----------+----------+------------+---------+--------+



In [87]:
spending.withColumn('prev_fee', F.lag('total_fees', 1).over(w_lead)).show()

+----------+----------+------------+---------+--------+
|patient_id|total_fees|patient_name|     city|prev_fee|
+----------+----------+------------+---------+--------+
|       101|      2500|Rahul Sharma|Hyderabad|    NULL|
|       102|      2000| Priya Reddy|Bangalore|    2500|
|       103|      1500|  Amit Kumar|   Mumbai|    2000|
|       104|      2500| Sneha Patel|  Chennai|    1500|
|       105|      1000|  Farhan Ali|    Delhi|    2500|
|       107|      2000| Arjun Verma|     Pune|    1000|
|       108|      NULL|  Meera Nair|    Kochi|    2000|
|       110|      2500| Nisha Reddy|Bangalore|    NULL|
|       120|      1500|        NULL|     NULL|    2500|
+----------+----------+------------+---------+--------+



In [92]:
from google.colab import files
uploaded = files.upload()

Saving patient_preferences.json to patient_preferences.json


In [93]:
prefs_df = spark.read.option('multiLine', True).json('patient_preferences.json')

In [94]:
prefs_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [95]:
prefs_df.select('patient_id', 'preferred_hospital',
F.col('contact.phone').alias('phone')).show()

+----------+------------------+----------+
|patient_id|preferred_hospital|     phone|
+----------+------------------+----------+
|       101|            Apollo|9876500011|
|       102|           Yashoda|      NULL|
|       103|              Care|9876500013|
|       104|              NULL|9876500014|
+----------+------------------+----------+



In [96]:
prefs_df.select('patient_id', 'preferred_hospital',
F.col('contact.email').alias('email')).show()

+----------+------------------+---------------+
|patient_id|preferred_hospital|          email|
+----------+------------------+---------------+
|       101|            Apollo|rahul@gmail.com|
|       102|           Yashoda|priya@gmail.com|
|       103|              Care|           NULL|
|       104|              NULL|sneha@gmail.com|
+----------+------------------+---------------+



In [97]:
prefs_df.filter(F.col('contact.phone').isNull()).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{priya@gmail.com,...|       102|           Yashoda|
+--------------------+----------+------------------+



In [98]:
prefs_df.filter(F.col('contact.email').isNull()).show()

+------------------+----------+------------------+
|           contact|patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|       103|              Care|
+------------------+----------+------------------+



In [99]:
prefs_df.filter(F.col('preferred_hospital').isNull()).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [100]:
prefs_flat = prefs_df.select(
'patient_id', 'preferred_hospital',
F.col('contact.phone').alias('phone'),
F.col('contact.email').alias('email')
)
prefs_flat.fillna({'phone': 'N/A'}).show()


+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       101|            Apollo|9876500011|rahul@gmail.com|
|       102|           Yashoda|       N/A|priya@gmail.com|
|       103|              Care|9876500013|           NULL|
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [101]:
prefs_flat.fillna({'email': 'no-email@example.com'}).show()

+----------+------------------+----------+--------------------+
|patient_id|preferred_hospital|     phone|               email|
+----------+------------------+----------+--------------------+
|       101|            Apollo|9876500011|     rahul@gmail.com|
|       102|           Yashoda|      NULL|     priya@gmail.com|
|       103|              Care|9876500013|no-email@example.com|
|       104|              NULL|9876500014|     sneha@gmail.com|
+----------+------------------+----------+--------------------+



In [102]:
prefs_flat.join(patients_df, on='patient_id', how='inner').show()

+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+
|patient_id|preferred_hospital|     phone|          email|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+
|       101|            Apollo|9876500011|rahul@gmail.com|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102|           Yashoda|      NULL|priya@gmail.com| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|              Care|9876500013|           NULL|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104|              NULL|9876500014|sneha@gmail.com| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+



In [103]:
patients_df.createOrReplaceTempView('patients')

In [104]:
appointments_df.createOrReplaceTempView('appointments')

In [105]:
spark.sql('SELECT * FROM patients').show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [106]:
spark.sql("SELECT * FROM patients WHERE city = 'Hyderabad'").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [107]:
spark.sql("SELECT * FROM patients WHERE city = 'Hyderabad'").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [108]:
spark.sql('SELECT department, COUNT(*) AS appt_count FROM appointments GROUP BY department').show()

+-----------+----------+
| department|appt_count|
+-----------+----------+
|  Neurology|         2|
|Dermatology|         3|
| Cardiology|         3|
|Orthopedics|         2|
+-----------+----------+



In [109]:
spark.sql('SELECT department, ROUND(AVG(consultation_fee),2) AS avg_fee FROM appointments GROUP BY department').show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [110]:
spark.sql('SELECT MAX(consultation_fee) AS max_fee FROM appointments').show()

+-------+
|max_fee|
+-------+
|   2500|
+-------+



In [111]:
spark.sql('''
    SELECT a.patient_id, p.patient_name, COUNT(a.appointment_id) AS appt_count
    FROM appointments a
    LEFT JOIN patients p ON a.patient_id = p.patient_id
    GROUP BY a.patient_id, p.patient_name
    ORDER BY appt_count DESC
''').show()

+----------+------------+----------+
|patient_id|patient_name|appt_count|
+----------+------------+----------+
|       101|Rahul Sharma|         2|
|       107| Arjun Verma|         1|
|       108|  Meera Nair|         1|
|       120|        NULL|         1|
|       110| Nisha Reddy|         1|
|       105|  Farhan Ali|         1|
|       104| Sneha Patel|         1|
|       103|  Amit Kumar|         1|
|       102| Priya Reddy|         1|
+----------+------------+----------+



In [112]:
spark.sql('''
    SELECT a.patient_id, p.patient_name, SUM(a.consultation_fee) AS total_spent
    FROM appointments a
    LEFT JOIN patients p ON a.patient_id = p.patient_id
    GROUP BY a.patient_id, p.patient_name
    ORDER BY total_spent DESC
    LIMIT 5
''').show()

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       110| Nisha Reddy|       2500|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       107| Arjun Verma|       2000|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+



In [113]:
etl_patients = spark.read.csv('patients.csv', header=True, inferSchema=True)
etl_appts    = spark.read.csv('appointments.csv', header=True, inferSchema=True)

In [114]:
etl_prefs = spark.read.option('multiLine', True).json('patient_preferences.json')
etl_prefs_flat = etl_prefs.select(
'patient_id', 'preferred_hospital',
 F.col('contact.phone').alias('phone'),
F.col('contact.email').alias('email'))


In [115]:
etl_patients = etl_patients.fillna({'city': 'Unknown', 'blood_group': 'Not Available'})
etl_appts    = etl_appts.fillna({'consultation_fee': 0})
etl_prefs_flat = etl_prefs_flat.fillna({'phone': 'N/A', 'email': 'N/A', 'preferred_hospital': 'Not Specified'})

In [116]:
joined_df = etl_patients \
    .join(etl_appts, on='patient_id', how='left') \
    .join(etl_prefs_flat, on='patient_id', how='left')
joined_df.show(truncate=False)

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+----------+---------------+
|patient_id|patient_name|city     |age|gender|blood_group  |insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |preferred_hospital|phone     |email          |
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+----------+---------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+           |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|Apollo            |9876500011|rahul@gmail.com|
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+           |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|Apollo      

In [117]:
joined_df = joined_df.withColumn(
      'age_group',
       F.when(F.col('age') < 30, 'Young')
      .when((F.col('age') >= 30) & (F.col('age') <= 50), 'Middle-aged')
      .otherwise('Senior')
  )
joined_df.select('patient_name', 'age', 'age_group').show()


+------------+---+-----------+
|patient_name|age|  age_group|
+------------+---+-----------+
|Rahul Sharma| 35|Middle-aged|
|Rahul Sharma| 35|Middle-aged|
| Priya Reddy| 29|      Young|
|  Amit Kumar| 42|Middle-aged|
| Sneha Patel| 31|Middle-aged|
|  Farhan Ali| 55|     Senior|
|  Neha Singh| 38|Middle-aged|
| Arjun Verma| 26|      Young|
|  Meera Nair| 48|Middle-aged|
|   Kiran Rao| 33|Middle-aged|
| Nisha Reddy| 41|Middle-aged|
+------------+---+-----------+



In [118]:
joined_df = joined_df.withColumn(
      'fee_category',
      F.when(F.col('consultation_fee') >= 2000, 'High')
       .when(F.col('consultation_fee') >= 1000, 'Medium')
      .otherwise('Low'))

joined_df.select('appointment_id', 'consultation_fee', 'fee_category').show()


+--------------+----------------+------------+
|appointment_id|consultation_fee|fee_category|
+--------------+----------------+------------+
|          5003|            1000|      Medium|
|          5001|            1500|      Medium|
|          5002|            2000|        High|
|          5004|            1500|      Medium|
|          5005|            2500|        High|
|          5006|            1000|      Medium|
|          NULL|            NULL|         Low|
|          5007|            2000|        High|
|          5010|               0|         Low|
|          NULL|            NULL|         Low|
|          5008|            2500|        High|
+--------------+----------------+------------+



In [119]:
joined_df = joined_df.withColumn(
      'fee_category',
      F.when(F.col('consultation_fee') >= 2000, 'High')
        .when(F.col('consultation_fee') >= 1000, 'Medium')
        .otherwise('Low'))
joined_df.select('appointment_id', 'consultation_fee', 'fee_category').show()


+--------------+----------------+------------+
|appointment_id|consultation_fee|fee_category|
+--------------+----------------+------------+
|          5003|            1000|      Medium|
|          5001|            1500|      Medium|
|          5002|            2000|        High|
|          5004|            1500|      Medium|
|          5005|            2500|        High|
|          5006|            1000|      Medium|
|          NULL|            NULL|         Low|
|          5007|            2000|        High|
|          5010|               0|         Low|
|          NULL|            NULL|         Low|
|          5008|            2500|        High|
+--------------+----------------+------------+



In [120]:
patient_spending = joined_df.groupBy('patient_id', 'patient_name').agg(
    F.sum('consultation_fee').alias('total_spending'),
    F.count('appointment_id').alias('total_appointments')
    ).orderBy('total_spending', ascending=False)
patient_spending.show()

+----------+------------+--------------+------------------+
|patient_id|patient_name|total_spending|total_appointments|
+----------+------------+--------------+------------------+
|       110| Nisha Reddy|          2500|                 1|
|       101|Rahul Sharma|          2500|                 2|
|       104| Sneha Patel|          2500|                 1|
|       107| Arjun Verma|          2000|                 1|
|       102| Priya Reddy|          2000|                 1|
|       103|  Amit Kumar|          1500|                 1|
|       105|  Farhan Ali|          1000|                 1|
|       108|  Meera Nair|             0|                 1|
|       109|   Kiran Rao|          NULL|                 0|
|       106|  Neha Singh|          NULL|                 0|
+----------+------------+--------------+------------------+



In [121]:
dept_revenue = joined_df.groupBy('department').agg(
      F.sum('consultation_fee').alias('total_revenue'),
      F.count('appointment_id').alias('total_appointments'),
      F.avg('consultation_fee').alias('avg_fee')
      ).orderBy('total_revenue', ascending=False)
dept_revenue.show()

+-----------+-------------+------------------+-----------------+
| department|total_revenue|total_appointments|          avg_fee|
+-----------+-------------+------------------+-----------------+
|Orthopedics|         5000|                 2|           2500.0|
|  Neurology|         4000|                 2|           2000.0|
| Cardiology|         3000|                 2|           1500.0|
|Dermatology|         2000|                 3|666.6666666666666|
|       NULL|         NULL|                 0|             NULL|
+-----------+-------------+------------------+-----------------+



In [122]:
joined_df.write.mode('overwrite').parquet('hospital_etl_output')

In [123]:
print('         HOSPITAL ANALYTICS REPORT')
print('\n--- Total Patients ---')
print(etl_patients.count())
print('\n--- Total Appointments ---')
print(etl_appts.count())
print('\n--- Appointments by Status ---')
etl_appts.groupBy('status').count().show()
print('\n--- Revenue by Department ---')
dept_revenue.show()
print('\n--- Top 5 Spending Patients ---')
patient_spending.limit(5).show()
print('\n--- Patient Age Group Distribution ---')
joined_df.groupBy('age_group').agg(
    F.countDistinct('patient_id').alias('patient_count')
    ).show()

print('\n--- Insurance Status Breakdown ---')
etl_patients.groupBy('insurance_status').count().show()

print("END OF REPORT")


         HOSPITAL ANALYTICS REPORT

--- Total Patients ---
10

--- Total Appointments ---
10

--- Appointments by Status ---
+---------+-----+
|   status|count|
+---------+-----+
|Completed|    7|
|Cancelled|    1|
|  Pending|    2|
+---------+-----+


--- Revenue by Department ---
+-----------+-------------+------------------+-----------------+
| department|total_revenue|total_appointments|          avg_fee|
+-----------+-------------+------------------+-----------------+
|Orthopedics|         5000|                 2|           2500.0|
|  Neurology|         4000|                 2|           2000.0|
| Cardiology|         3000|                 2|           1500.0|
|Dermatology|         2000|                 3|666.6666666666666|
|       NULL|         NULL|                 0|             NULL|
+-----------+-------------+------------------+-----------------+


--- Top 5 Spending Patients ---
+----------+------------+--------------+------------------+
|patient_id|patient_name|total_spendin